# CoffeeGuard — Notebook 02: Benchmark nhiều kiến trúc

**Bật GPU:** `Runtime` → `Change runtime type` → `T4 GPU`.

Notebook này **không tự chia lại tập, không tự định nghĩa thứ tự lớp, không tự
resize ảnh**. Cả ba đã khoá ở notebook 01 và đi kèm trong gói dữ liệu.

| Xuất ra | Dùng để |
|---|---|
| `reports/predictions/pred__*.csv` | notebook 03 chấm — **một file / một lần chạy** |
| `models/onnx/*.onnx` | thả vào `web/public/models/` cho PWA |
| `reports/latency.json` | cột tốc độ trong bảng so sánh |
| `reports/registry_moi.json` | ghép vào `registry.json` của PWA |

---

### Ba điều notebook này **cố tình không làm**

1. **Không early-stopping theo fold test.** Chọn epoch tốt nhất bằng chính tập
   test là rò rỉ: chỉ số đẹp lên vài phần trăm và không ai thấy gì bất thường.
   Ở đây dùng **số epoch cố định** cho mọi kiến trúc.
2. **Không chạy nhiều seed.** Phương sai lấy từ chênh lệch giữa 5 fold. Thêm
   seed chỉ tốn giờ GPU mà không trả lời thêm câu hỏi nào.
3. **Không trộn BRACOL vào tập đánh giá.** BRACOL khác miền hoàn toàn (lá ngắt
   rời / nền trắng / studio / mặt dưới). Trộn thẳng thì model học *"ảnh này
   thuộc dataset nào"* — dễ hơn học bệnh — mà severity lại phân bố khác nhau
   giữa hai dataset, thành biến gây nhiễu. Chỉ số vẫn đẹp, kết luận vẫn sai.

## 0. Khởi động

In [ ]:
# --- O KHOI DONG: chay duoc CA tren Colab LAN tren laptop ---
import os, sys, json, time, zipfile, hashlib, subprocess
from pathlib import Path

try:
    import google.colab  # noqa: F401
    TREN_COLAB = True
except ImportError:
    TREN_COLAB = False

if TREN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)
    GOC = Path("/content/CoffeeGuard")
    GOC.mkdir(parents=True, exist_ok=True)
    os.environ["COFFEEGUARD_ROOT"] = str(GOC)

    # Ma nguon hop dong (~40 KB). Tim theo thu tu: da giai nen > Drive > upload.
    if not (GOC / "src" / "coffeeguard" / "bench" / "contract.py").exists():
        nguon = Path("/content/drive/MyDrive/CoffeeGuard/coffeeguard_src.zip")
        if not nguon.exists():
            print("Chon file coffeeguard_src.zip (o §0 cua notebook 01 tren laptop sinh ra):")
            from google.colab import files
            up = files.upload()
            nguon = Path(next(iter(up)))
        with zipfile.ZipFile(nguon) as z:
            z.extractall(GOC)
        print("da giai nen ma nguon ->", GOC / "src")
    SRC = GOC / "src"
else:
    # Notebook nam o <repo>/notebooks/ nen goc repo la thu muc cha.
    GOC = Path.cwd()
    while not (GOC / "pyproject.toml").exists() and GOC != GOC.parent:
        GOC = GOC.parent
    SRC = GOC / "src"

sys.path.insert(0, str(SRC))

from coffeeguard.bench import contract as ct
from coffeeguard.bench import paths as P

assert P.ROOT == GOC, f"Goc du an khong khop: {P.ROOT} != {GOC}"
print("moi truong :", "Google Colab" if TREN_COLAB else "laptop")
print("goc du an  :", P.ROOT)
print("classes_6  :", list(ct.CLASSES_6))

## 1. Cấu hình

Dán `sha256` in ra ở §5 của notebook 01 vào `SHA256_GOI`.

In [ ]:
# ============================ CHINH O DAY ============================
GOI            = "coffeeguard_bench_leafsquare-ngan1.4x_224.zip"
SHA256_GOI     = "3fe41b95571584275b7a228f319b37626dd17c769e91b133659902d92153fd4e"  # sinh 30/09; de "" = BO QUA kiem chung
#                  ^ khop file data/coffeeguard_bench_*.zip.sha256.
#                    Doi goi (dong goi lai) thi PHAI cap nhat dong nay,
#                    khong thi cell §2 dung lai voi loi SHA256 KHONG KHOP.
THU_MUC_DRIVE  = "MyDrive/CoffeeGuard"

# DANH SACH DA CHOT 30/09/2026 — xem Docs/Ke_hoach_benchmark.md §4.
# Ca 5 deu thoa dang thuc CAM NGUYEN BAN (da kiem bang hanh vi tren 33 ung vien).
# DUNG them kien truc vao day ma khong chay: python scripts/kiem_cam.py <ten>
# Ba kien truc cua ban nhap cu (mobilenetv3_small_100, mobilenetv4_conv_small,
# efficientvit_b0) DA BI LOAI vi co phi tuyen giua GAP va classifier.
KIEN_TRUC = [
    # (ten timm, bi danh ghi vao bang)
    ("mobilenetv2_100",    "mobilenetv2"),      # san dung luong + baseline mobile duoc dan nhieu nhat
    ("mobilevitv2_050",    "mobilevitv2_050"),  # dai dien lai attention
    ("efficientnet_lite0", "effnet_lite0"),     # ban cho mobile: khong SE, khong hardswish
    ("efficientnet_b0",    "effnet_b0"),        # moc tham chieu pho bien nhat
    ("resnet18",           "resnet18"),         # baseline co dien
]

EPOCHS       = 15      # co dinh cho MOI kien truc — xem ly do o o mo dau
BATCH        = 32
LR           = 3e-4
WEIGHT_DECAY = 1e-4
LABEL_SMOOTH = 0.05
SEED         = 42
CHAY_FOLD    = [0, 1, 2, 3, 4]   # rut ngan khi chay thu, vd [0]
RUN_ID       = time.strftime("%Y%m%d-%H%M")
# =====================================================================

print("goi   :", GOI)
print("run_id:", RUN_ID)
print("fold  :", CHAY_FOLD)
print("arch  :", ", ".join(b for _, b in KIEN_TRUC))


## 2. Nạp gói + kiểm chứng

Thứ tự kiểm: file tồn tại → sha256 khớp → giải nén → **thứ tự lớp trong file
split khớp `contract.py`**. Bước cuối dễ bị bỏ qua nhất và có hậu quả tệ nhất.

In [ ]:
def bam(p, chunk=1 << 20):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        while blk := f.read(chunk):
            h.update(blk)
    return h.hexdigest()

if TREN_COLAB:
    nguon = Path("/content/drive") / THU_MUC_DRIVE / GOI
    if not nguon.exists():
        raise FileNotFoundError(
            f"Khong thay {nguon}.\nChay notebook 01 roi tai file .zip len Drive."
        )
    h = bam(nguon)
    print("sha256 goi:", h)
    if SHA256_GOI and h != SHA256_GOI:
        raise ValueError(
            "SHA256 KHONG KHOP — goi tai len hong hoac khong phai ban moi nhat.\n"
            f"  tren dia : {h}\n  mong doi : {SHA256_GOI}\n"
            "DUNG LAI. Train tren goi hong thi moi con so deu vo nghia ma van chay tron tru."
        )
    if not SHA256_GOI:
        print("[CANH BAO] chua dat SHA256_GOI -> khong kiem chung duoc tinh toan ven.")
    if not (GOC / "data" / "splits").exists():
        with zipfile.ZipFile(nguon) as z:
            z.extractall(GOC)
        print("da giai nen ->", GOC)
else:
    print("chay local — dung thang du lieu trong repo, khong can goi .zip")

from coffeeguard.bench import index as bidx
from coffeeguard.bench import splits as bsplit
from coffeeguard.bench import predictions as bpred
from coffeeguard.bench import latency as blat

meta  = bsplit.doc_split()              # tu no assert thu tu lop khop contract
index = bsplit.gan_fold(bidx.doc_index(), meta)

# Chon thu muc anh THEO TEN, khong lay "thu muc dau tien tim thay": co ca bo
# leafsquare lan bo letterbox, lay nham thi van train tron tru tren bo kia.
ten_bo = Path(GOI).stem.replace("coffeeguard_bench_", "")
THU_MUC_ANH = GOC / "data" / "processed" / ten_bo
if not THU_MUC_ANH.exists():
    co = [d.name for d in (GOC / "data" / "processed").iterdir() if d.is_dir()]
    raise FileNotFoundError(f"Khong thay bo anh '{ten_bo}'. Dang co: {co}")
manifest = json.loads((THU_MUC_ANH / "manifest.json").read_text(encoding="utf-8"))

print("\n--- san sang ---")
print("  bo anh     :", THU_MUC_ANH.name, "|", sum(1 for _ in THU_MUC_ANH.glob("*.png")), "PNG")
print("  che do cat :", manifest["che_do_cat"], "| neo", manifest.get("leaf_neo"),
      "| scale", manifest["leaf_scale"])
print("  la chiem   :", manifest["ti_le_pixel_la"])
print("  anh/cay/la :", meta["n_images"], meta["n_plants"], meta["n_leaves"])
print("  split seed :", meta["seed"], "|", meta["thuat_toan"])
for c in meta["canh_bao"]:
    print("  [CANH BAO]", c)

## 3. Thư viện

In [ ]:
def cai(*pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

# GHIM PHIEN BAN — vi sao:
#   Cau truc DAU model phu thuoc phien ban timm. Bo loc CAM (§4 ke hoach) da duoc
#   kiem chung tren timm 1.0.30; mot ban khac co the doi thu tu lop trong head,
#   lam dang thuc CAM vo ma KHONG bao loi. Colab `pip install timm` lay ban moi
#   nhat, nen phai ghim. O §3b ngay duoi se kiem lai bang HANH VI — ghim chi la
#   lop bao ve thu nhat, phep kiem moi la lop that.
TIMM_GHIM = "timm==1.0.30"

try:
    import timm
    if timm.__version__ != TIMM_GHIM.split("==")[1]:
        print(f"[CANH BAO] timm {timm.__version__} != ban da kiem chung "
              f"{TIMM_GHIM.split('==')[1]}. O §3b se kiem lai bang hanh vi.")
except ImportError:
    cai(TIMM_GHIM); import timm
# KHONG cai onnx / onnxruntime tren Colab — CO Y.
#   Colab ghim protobuf 5.29.6, con `onnx` moi duoc build voi gencode 6.31.1
#   -> import la ne VersionError. Nang protobuf len 6.x thi pha `grpcio-status`
#   (ghim <6.0dev) va `google-ai-generativelanguage`. Ha `onnx` xuong ban cu chi
#   la phong doan, co the vo lai khi Colab doi anh nen.
#   => Colab CHI train. Export ONNX lam tren laptop bang `scripts/export_onnx.py`,
#      noi onnx va protobuf da khop nhau. Xem §9.

import torch, numpy as np, pandas as pd
pd.set_option("display.width", 200)
print("torch", torch.__version__, "| cuda", torch.cuda.is_available())
print("timm", timm.__version__)

# Ghi lai moi truong vao bien de dong goi ket qua mang ve (truy nguyen duoc)
MOI_TRUONG = {"torch": torch.__version__, "timm": timm.__version__,
              "python": sys.version.split()[0],
              "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None}

DEV = "cuda" if torch.cuda.is_available() else "cpu"
if DEV == "cpu":
    print()
    print("=" * 66)
    print("[CANH BAO] DANG CHAY CPU — KHONG PHAI GPU")
    print("  5 kien truc x 5 fold tren CPU mat NHIEU GIO thay vi 1-2 gio.")
    print("  Runtime > Change runtime type > T4 GPU > Save.")
    print("  LUU Y: doi runtime se KHOI DONG LAI may ao va xoa moi thu da cai,")
    print("         nen hay doi TRUOC roi chay lai notebook tu dau.")
    print("=" * 66)


## 4. Dataset + augmentation

### Vì sao `RandomResizedCrop` chứ không để nguyên

Ảnh trong gói đã là **cửa sổ vuông neo cạnh ngắn 1,4×**, tức lá chiếm ~61% khung
và **hai đầu lá đã bị cắt**. Hai hệ quả, giải cùng một cách:

1. Lúc chạy thật, nông dân được PWA dặn *"đưa lá vào khung"* nên lá chiếm 70%+.
   Model train một tỉ lệ mà chạy ở tỉ lệ khác thì tụt chính xác **mà không có gì
   báo**.
2. Cửa sổ cố định luôn rơi vào **giữa** lá, nên model không bao giờ thấy phần
   gần chóp.

→ `RandomResizedCrop(scale=(0.45, 1.0))` vừa trải rộng tỉ lệ lá quanh khoảng lúc
chạy thật, vừa làm mỗi epoch ô cắt rơi vào một đoạn khác của lá.

### Vì sao lật cả dọc lẫn ngang

Lá cà phê không có chiều "đúng"; ảnh RoCoLe chụp cầm tay, hướng tuỳ ý.

### Vì sao `mean`/`std` lấy theo từng model

Quy tắc 11. ViT và MobileNet chuẩn hoá khác nhau. Dùng chung một pipeline thì
một model nhận đầu vào lệch phân bố, trông kém đi vài phần trăm, và ta kết luận
sai rằng *"kiến trúc này kém"*. Giá trị lấy từ
`timm.data.resolve_model_data_config` và được ghi vào `registry.json` để PWA
dùng **đúng** con số đó.

In [ ]:
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image

LOP2ID = {c: i for i, c in enumerate(ct.CLASSES_6)}

class BoDuLieuLa(Dataset):
    def __init__(self, df_, thu_muc, bien_doi):
        self.png = df_.png.tolist()
        self.y = [LOP2ID[c] for c in df_.label6]
        self.thu_muc = Path(thu_muc)
        self.bien_doi = bien_doi
    def __len__(self):
        return len(self.png)
    def __getitem__(self, i):
        with Image.open(self.thu_muc / self.png[i]) as im:
            return self.bien_doi(im.convert("RGB")), self.y[i]

def bien_doi_train(size, mean, std):
    return T.Compose([
        T.RandomResizedCrop(size, scale=(0.45, 1.0), ratio=(0.85, 1.18),
                            interpolation=T.InterpolationMode.BICUBIC),
        T.RandomHorizontalFlip(),
        T.RandomVerticalFlip(),
        T.RandomApply([T.ColorJitter(0.25, 0.25, 0.20, 0.03)], p=0.7),
        T.ToTensor(), T.Normalize(mean, std),
    ])

def bien_doi_test(size, mean, std):
    # KHONG augmentation, KHONG crop. Anh da 224 va da khoa tien xu ly.
    ops = [] if size == 224 else [T.Resize((size, size), T.InterpolationMode.BICUBIC)]
    return T.Compose(ops + [T.ToTensor(), T.Normalize(mean, std)])

def trong_so_lop(df_):
    """1/sqrt(n) — giua 'khong can bang gi' va 'oversample thang'.

    Oversample theo 1/n day rust_level_4 (29 anh) lap lai ~27 lan moi epoch;
    model thuoc long dung 29 tam do, recall tren train vot len con test thi
    khong. 1/sqrt(n) giam bot ma van keo lop hiem ra khoi vung bi bo qua.
    """
    n = df_.label6.value_counts().reindex(list(ct.CLASSES_6)).fillna(0).to_numpy()
    w = 1.0 / np.sqrt(np.maximum(n, 1))
    return torch.tensor(w / w.mean(), dtype=torch.float32)

display(pd.crosstab(index.label6, index.fold).reindex(list(ct.CLASSES_6)))

## 5. Model + CAM tính sẵn trong graph

`onnxruntime-web` **chỉ suy luận, không có gradient**, nên Grad-CAM không chạy
được trong trình duyệt. Thay bằng **CAM gốc** (Zhou 2016):

```
cam[k] = conv1x1(feature_map, W_classifier[k])
```

Nhúng phép đó vào graph ONNX lúc export thì JS chỉ việc lấy kênh của lớp được
dự đoán — không phải chuyển trọng số phân loại xuống client.

**Ràng buộc:** kiến trúc phải kết thúc `GAP → một Linear` *và* số kênh feature
map phải bằng `in_features` của Linear đó.

`mobilenetv3` **không thoả**: head của nó là `GAP → conv_head → act → Linear`.
Lớp `ModelCAM` phát hiện, báo rõ, rồi cho model đó chạy tiếp **không có CAM** —
nó vẫn dự thi phần phân loại, chỉ mất bản đồ nhiệt. Đây là **một kết quả của đồ
án**: ràng buộc *"hiển thị trực quan vùng tổn thương"* trong thuyết minh loại
bớt kiến trúc nào.

In [ ]:
import torch.nn as nn
from coffeeguard.bench import cam as bcam


class ModelCAM(nn.Module):
    """timm backbone -> (logits, cam).

    Kiem tinh hop le cua CAM bang HANH VI (`coffeeguard.bench.cam`), khong bang
    ten module hay so kenh. Vi sao: kiem bang so kenh cho DUONG TINH GIA voi ho
    ConvNeXt/EdgeNeXt — so kenh khop, lop cuoi dung la nn.Linear, nhung co chuan
    hoa toan cuc sau pooling nen dang thuc CAM vo. Da do: convnext_tiny sai so
    1.8e-01 ma kiem cau truc van noi "co CAM". Xem cam.py va §4 ke hoach.
    """

    def __init__(self, ten_timm, n_lop, pretrained=True):
        super().__init__()
        self.ten = ten_timm
        self.backbone = timm.create_model(ten_timm, pretrained=pretrained, num_classes=n_lop)
        self.head = self.backbone.get_classifier()
        # kiem_cam TU dua model ve eval() roi tra lai che do cu. Bat buoc, vi
        # timm.create_model tra model o che do TRAIN, va o che do do BatchNorm
        # dung thong ke theo batch (phi tuyen theo dau vao) -> phep thu cong
        # tinh cho AM TINH GIA. Da mac that: mobilevitv2_050 bao 1.4e-01 o che
        # do train nhung 4.0e-08 o eval. Khong truyen kich thuoc -> lay kich
        # thuoc GOC cua kien truc (mobilevitv2_050 chay 256, khong phai 224).
        kq = bcam.kiem_cam(self.backbone)
        self.co_cam = kq["co_cam"]
        self.ly_do = kq["ly_do"]
        self.sai_so_cam = kq["sai_so"]
        self.kich_thuoc_kiem = kq["kich_thuoc"]
        self.cuu_duoc = kq["cuu_duoc"]

    def forward(self, x):
        f = self.backbone.forward_features(x)
        logits = self.backbone.forward_head(f)
        if not self.co_cam:
            return logits
        return logits, torch.einsum("bchw,kc->bkhw", f, self.head.weight)


def tao_model(ten_timm, pretrained=True):
    m = ModelCAM(ten_timm, len(ct.CLASSES_6), pretrained=pretrained)
    cfg = timm.data.resolve_model_data_config(m.backbone)
    return m, {
        "ten_timm": ten_timm,
        "input_size": list(cfg["input_size"]),
        "mean": [round(float(v), 6) for v in cfg["mean"]],
        "std":  [round(float(v), 6) for v in cfg["std"]],
        "interpolation": cfg.get("interpolation", "bicubic"),
        "co_cam": m.co_cam,
        "sai_so_cam": m.sai_so_cam,
        "kich_thuoc_kiem": m.kich_thuoc_kiem,
        "ly_do_khong_cam": m.ly_do,
        **blat.dem_tham_so(m),
    }


# --- CHAN TRUOC KHI TRAIN: tha dung ngay con hon train 5 fold roi moi biet ---
# Nem RuntimeError neu bat ky kien truc nao mat CAM. Day la lop bao ve THAT
# (khac voi viec ghim phien ban timm o §3, chi la lop thu nhat).
_ = bcam.chan_neu_mat_cam([t for t, _b in KIEN_TRUC], len(ct.CLASSES_6))
print("OK — ca", len(KIEN_TRUC), "kien truc deu thoa dang thuc CAM tren timm",
      timm.__version__)

SOI = {}
hang = []
for ten, bi_danh in KIEN_TRUC:
    _m, tt = tao_model(ten, pretrained=False)
    SOI[bi_danh] = tt
    hang.append({"arch": bi_danh, "params_M": tt["params_M"],
                 "size_MB_fp32": tt["size_MB_fp32"], "input": tt["input_size"][1],
                 "CAM": "co" if tt["co_cam"] else "KHONG",
                 "sai_so_cam": f"{tt['sai_so_cam']:.1e}" if tt["sai_so_cam"] is not None else "-",
                 "kiem_o": tt["kich_thuoc_kiem"],
                 "ly_do": tt["ly_do_khong_cam"][:50]})
    del _m
display(pd.DataFrame(hang))


## 6. Vòng huấn luyện 5 fold

Mỗi fold: train trên 4 fold còn lại, dự đoán fold này. Mỗi ảnh vào test **đúng
một lần** → gộp lại đủ 1.559 dự đoán, trong đó `rust_level_4` có 29 thay vì 3.

In macro-F1 của từng fold ngay khi xong: fold nào lệch hẳn khỏi các fold còn lại
là dấu hiệu **train hỏng**, không phải "phương sai tự nhiên".

In [ ]:
from sklearn.metrics import f1_score

def dat_seed(s):
    import random
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)

@torch.inference_mode()
def du_doan(model, loader):
    model.eval()
    ra = []
    for x, _ in loader:
        out = model(x.to(DEV, non_blocking=True))
        logits = out[0] if isinstance(out, tuple) else out
        ra.append(torch.softmax(logits.float(), dim=1).cpu().numpy())
    return np.concatenate(ra)

def train_mot_fold(ten_timm, fold, tt, epochs=EPOCHS):
    dat_seed(SEED + fold)
    size = tt["input_size"][1]
    tr_df, te_df = index[index.fold != fold], index[index.fold == fold]

    dl_tr = DataLoader(BoDuLieuLa(tr_df, THU_MUC_ANH, bien_doi_train(size, tt["mean"], tt["std"])),
                       batch_size=BATCH, shuffle=True, num_workers=2,
                       pin_memory=True, drop_last=True)
    dl_te = DataLoader(BoDuLieuLa(te_df, THU_MUC_ANH, bien_doi_test(size, tt["mean"], tt["std"])),
                       batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

    model, _ = tao_model(ten_timm, pretrained=True)
    model.to(DEV)
    lf = nn.CrossEntropyLoss(weight=trong_so_lop(tr_df).to(DEV), label_smoothing=LABEL_SMOOTH)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    sch = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, epochs=epochs,
                                              steps_per_epoch=len(dl_tr), pct_start=0.25)
    scaler = torch.amp.GradScaler(enabled=(DEV == "cuda"))

    t0 = time.time()
    for ep in range(epochs):
        model.train(); tong = 0.0
        for x, y in dl_tr:
            x, y = x.to(DEV, non_blocking=True), y.to(DEV, non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEV, enabled=(DEV == "cuda")):
                out = model(x)
                loss = lf(out[0] if isinstance(out, tuple) else out, y)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update(); sch.step()
            tong += loss.item() * x.size(0)
        print(f"    ep{ep+1:02d}/{epochs} loss {tong/len(dl_tr.dataset):.4f}", end="\r")

    Pm = du_doan(model, dl_te)
    y_true = np.array([LOP2ID[c] for c in te_df.label6])
    f1 = f1_score(y_true, Pm.argmax(1), average="macro", zero_division=0)
    print(f"    fold {fold}: macro-F1 = {f1:.4f}   ({time.time()-t0:.0f}s)          ")
    return model, te_df, Pm, f1

def train_toan_bo(ten_timm, bi_danh):
    tt = SOI[bi_danh]
    print(f"\n=== {bi_danh} ({ten_timm}) ===")
    manh, f1s, model_cuoi = [], [], None
    for fold in CHAY_FOLD:
        model, te_df, Pm, f1 = train_mot_fold(ten_timm, fold, tt)
        f1s.append(f1); model_cuoi = model
        sub = te_df[["png", "fold", "plant_id", "leaf_id", "face"]].copy()
        sub["y_true"] = te_df.label6.to_numpy()
        sub["y_pred"] = [ct.CLASSES_6[i] for i in Pm.argmax(1)]
        for j, c in enumerate(ct.CLASSES_6):
            sub[f"p_{c}"] = Pm[:, j]
        manh.append(sub)
        if fold != CHAY_FOLD[-1]:
            del model; torch.cuda.empty_cache()
    pred = pd.concat(manh, ignore_index=True)
    sd = np.std(f1s, ddof=1) if len(f1s) > 1 else 0.0
    print(f"  macro-F1 tung fold: {[round(x,4) for x in f1s]}")
    print(f"  trung binh {np.mean(f1s):.4f} +/- {sd:.4f}")
    return pred, model_cuoi, f1s

## 7. Chạy — và ghi file dự đoán đúng hợp đồng

`bpred.ghi_du_doan()` **kiểm chứng trước khi ghi**. Nó chặn đúng những lỗi mà
phía train không bao giờ ném exception:

| Lỗi | Bị bắt bằng |
|---|---|
| tự chia lại fold | cột `fold` lệch file split đã đóng băng |
| giải mã sai thứ tự lớp | `argmax(p_*)` ≠ `y_pred` |
| quên `softmax` | tổng xác suất ≠ 1 |
| thiếu / thừa ảnh | n ≠ số ảnh trong index |
| ghi đè kết quả cũ | trùng `run_id` → từ chối |

Chạy `CHAY_FOLD = [0]` một lượt trước cho nhanh, chắc chắn đường ống thông, rồi
mới mở đủ 5 fold.

In [ ]:
# BEN HOA NGAY sau MOI kien truc — khong doi den cuoi.
#
# Vi sao: /content la dia cua MAY AO, khong phai Drive. Colab hay rot ket noi va
# thu hoi may ao. Neu chi luu o cuoi thi mat ket noi o kien truc thu 4 la mat
# sach cong cua ca 4. Da xay ra that 30/09/2026.
# Moi kien truc xong: ghi pred -> luu .pt -> CHEP SANG DRIVE. Mat may ao luc nao
# cung chi mat dung kien truc dang chay do.

import shutil

MO_HINH_CUOI, LATENCY, DU_DOAN = {}, {}, {}
PT_DIR = GOC / "models" / "pt"
PT_DIR.mkdir(parents=True, exist_ok=True)
SAO_LUU = (Path("/content/drive") / THU_MUC_DRIVE / f"ketqua_tam_{RUN_ID}") if TREN_COLAB else None


def ben_hoa(duong_dan_tuong_doi):
    """Chep mot file sang Drive, giu nguyen cay thu muc. Loi thi canh bao, khong dung."""
    if SAO_LUU is None:
        return
    try:
        d = SAO_LUU / duong_dan_tuong_doi
        d.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(GOC / duong_dan_tuong_doi, d)
    except Exception as e:
        print(f"  [CANH BAO] khong chep duoc {duong_dan_tuong_doi} sang Drive: {e}")


for ten_timm, bi_danh in KIEN_TRUC:
    pred, model, f1s = train_toan_bo(ten_timm, bi_danh)
    DU_DOAN[bi_danh] = pred
    MO_HINH_CUOI[bi_danh] = model

    if len(CHAY_FOLD) == meta["n_folds"]:
        p = bpred.ghi_du_doan(pred, framework="pytorch-timm", arch=bi_danh, run_id=RUN_ID)
        print(f"  da ghi {p.name}")
        ben_hoa(f"reports/predictions/{p.name}")

        # Luu trong so NGAY, khong doi §9. Model dang o GPU -> chep ban CPU.
        pt = PT_DIR / f"{bi_danh}_{RUN_ID}.pt"
        torch.save({k: v.detach().cpu() for k, v in model.backbone.state_dict().items()}, pt)
        ben_hoa(f"models/pt/{pt.name}")

        # HO SO KEM: moi .pt di kem mot .json mo ta kien truc cua chinh no.
        # Nho vay §9 doc tu DIA chu khong tu bo nho. Neu mat ket noi va noi lai
        # voi KIEN_TRUC rut gon thi MO_HINH_CUOI chi con vai cai, va §9 se AM
        # THAM bo sot nhung kien truc da train truoc do — script export tren
        # laptop van chay tron tru, van bao "OK", chi la thieu model.
        ho_so = PT_DIR / f"{bi_danh}_{RUN_ID}.json"
        ho_so.write_text(json.dumps({
            "bi_danh": bi_danh, "ten_timm": SOI[bi_danh]["ten_timm"],
            "pt": f"models/pt/{pt.name}",
            "co_cam": SOI[bi_danh]["co_cam"], "sai_so_cam": SOI[bi_danh]["sai_so_cam"],
            "kich_thuoc_kiem": SOI[bi_danh]["kich_thuoc_kiem"],
            "input_size": SOI[bi_danh]["input_size"],
            "mean": SOI[bi_danh]["mean"], "std": SOI[bi_danh]["std"],
            "interpolation": SOI[bi_danh]["interpolation"],
            "params_M": SOI[bi_danh]["params_M"],
            "size_MB_fp32": SOI[bi_danh]["size_MB_fp32"],
            "macro_f1_6lop_tung_fold": [round(float(x), 4) for x in f1s],
            "latency_p50_ms_colab_cpu_1luong": None,
        }, ensure_ascii=False, indent=2), encoding="utf-8")
        ben_hoa(f"models/pt/{ho_so.name}")

        print(f"  da luu {pt.name} ({pt.stat().st_size/1048576:.1f} MB)"
              + (f" + sao luu Drive" if SAO_LUU else ""))
    else:
        print(f"  [CHAY THU {len(CHAY_FOLD)}/{meta['n_folds']} fold] khong ghi file du doan —")
        print("  file chi hop le khi du 5 fold, vi moi anh phai vao test dung mot lan.")

if SAO_LUU:
    print()
    print(f"ban sao an toan o: {SAO_LUU}")


## 8. Đo tốc độ

Latency **trên CPU của Colab**: warmup 20, chạy 50, lấy **trung vị**, **cố định
1 luồng**. Thiếu ba thứ đó thì bảng "tốc độ" phản ánh thứ tự nạp model và lịch
chạy của hệ điều hành chứ không phải kiến trúc.

Đây **không phải** số trên điện thoại. Số đó lấy từ PWA
(`web/src/lib/infer.js` trả `latencyMs` quanh đúng `session.run()`), đo trên máy
thật rồi mới được ghi vào `registry.json`.

In [ ]:
x_cpu = torch.randn(1, 3, 224, 224)
hang = []
for bi_danh, model in MO_HINH_CUOI.items():
    m = model.to("cpu").eval()
    kq = blat.do_model_torch(m, x_cpu, ten=bi_danh, n_warmup=20, n_chay=50, so_luong=1,
                             ghi_chu="PyTorch fp32, CPU Colab, 1 luong — KHONG phai so tren dien thoai")
    LATENCY[bi_danh] = kq.dict()
    # ghi nguoc vao ho so kem, de §9 doc duoc tu dia du co noi lai phien khac
    _hs = GOC / "models" / "pt" / f"{bi_danh}_{RUN_ID}.json"
    if _hs.exists():
        _d = json.loads(_hs.read_text(encoding="utf-8"))
        _d["latency_p50_ms_colab_cpu_1luong"] = kq.p50_ms
        _hs.write_text(json.dumps(_d, ensure_ascii=False, indent=2), encoding="utf-8")
    hang.append({"arch": bi_danh, "p50_ms": kq.p50_ms, "p90_ms": kq.p90_ms,
                 "FPS_p50": kq.fps_p50, "params_M": SOI[bi_danh]["params_M"]})
    model.to(DEV)
display(pd.DataFrame(hang).sort_values("p50_ms"))
print("thiet bi:", blat.mo_ta_may())

## 9. Luu trong so (export ONNX lam tren laptop)


In [ ]:
# §9 — GOM HO SO DE EXPORT. Export ONNX lam tren LAPTOP, khong lam o day.
#
# Vi sao khong export tai Colab: Colab ghim protobuf 5.29.6 nhung `onnx` moi can
# gencode 6.31.1 -> import ne VersionError. Ma torch.onnx.export TU import onnx
# ben trong, nen bo `import onnx` cung khong ne duoc. Nang protobuf thi pha
# grpcio-status (<6.0dev). Laptop da co onnx+protobuf khop nhau.
#
# O DAY DOC TU DIA, khong doc tu MO_HINH_CUOI trong bo nho.
# Ly do: neu mat ket noi giua chung va noi lai voi KIEN_TRUC rut gon thi bo nho
# chi con vai kien truc, con dia thi con DU. Doc tu bo nho se AM THAM bo sot —
# script export van chay, van bao OK, chi la thieu model.

PT_DIR = GOC / "models" / "pt"
PT_DIR.mkdir(parents=True, exist_ok=True)

ho_so_list = sorted(PT_DIR.glob(f"*_{RUN_ID}.json"))
DE_EXPORT = {"run_id": RUN_ID, "moi_truong": MOI_TRUONG,
             "classes": list(ct.CLASSES_6), "models": []}

for hs in ho_so_list:
    d = json.loads(hs.read_text(encoding="utf-8"))
    pt = GOC / d["pt"]
    if not pt.exists():
        print(f"[BO QUA] {hs.name}: khong thay {d['pt']}")
        continue
    d["size_MB_pt"] = round(pt.stat().st_size / 1048576, 2)
    if d.get("latency_p50_ms_colab_cpu_1luong") is None:
        d["latency_p50_ms_colab_cpu_1luong"] = LATENCY.get(d["bi_danh"], {}).get("p50_ms")
    DE_EXPORT["models"].append(d)
    print(f"{d['bi_danh']:18s} {d['size_MB_pt']:6.1f} MB  "
          f"CAM {'co' if d['co_cam'] else 'KHONG'}  "
          f"F1 tung fold {d.get('macro_f1_6lop_tung_fold')}")

(GOC / "reports").mkdir(exist_ok=True)
for ten, obj in (("de_export", DE_EXPORT), ("latency", LATENCY),
                 ("model_data_config", SOI)):
    (GOC / "reports" / f"{ten}.json").write_text(
        json.dumps(obj, ensure_ascii=False, indent=2), encoding="utf-8")

n = len(DE_EXPORT["models"])
print()
print(f"de_export.json: {n} model")
if n < 5:
    print(f"[CANH BAO] moi co {n}/5 kien truc. Neu con thieu thi chay not roi chay lai o nay,")
    print("           dung export khi chua du — bang so sanh se thieu kien truc.")
print()
print("BUOC TIEP THEO tren LAPTOP, sau khi giai nen goi ket qua vao goc repo:")
print("    python scripts/export_onnx.py")


## 10. Đóng gói mang về laptop

Cây thư mục trong zip giữ nguyên layout repo → giải nén **đè lên gốc repo** là
xong, không phải chép tay từng file.

In [ ]:
import shutil
ket_qua = Path("/content" if TREN_COLAB else GOC) / f"coffeeguard_ketqua_{RUN_ID}.zip"
with zipfile.ZipFile(ket_qua, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
    for thu_muc in ("reports", "models"):
        for p in sorted((GOC / thu_muc).rglob("*")):
            if p.is_file():
                z.write(p, str(p.relative_to(GOC)).replace("\\", "/"))

if TREN_COLAB:
    try:
        shutil.copy2(ket_qua, Path("/content/drive") / THU_MUC_DRIVE / ket_qua.name)
        print("da chep vao Drive")
    except Exception as e:
        print("khong chep duoc vao Drive:", e)

print(f"{ket_qua}  ({ket_qua.stat().st_size/1048576:.1f} MB)")
print("sha256:", bam(ket_qua))
for n in zipfile.ZipFile(ket_qua).namelist():
    print("   ", n)

## Việc tiếp theo

1. Tải `coffeeguard_ketqua_*.zip` về, giải nén **đè lên gốc repo**
2. Mở `03_cham_diem.ipynb`

### Ablation còn nợ

Chạy lại toàn bộ với gói `letterbox` (notebook 01 §5 đổi `CHE_DO_DONG_GOI`) và
đổi `RUN_ID`. Không có ablation thì *"cửa sổ vuông quanh lá giúp ích"* mới chỉ
là một phép đếm pixel, chưa phải kết quả.